PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict


In [2]:
Ai = 0 # A player of Team A
RAi = 0 # Rating of player Ai
RA  = 0 # Average elo rating of Team A
RB  = 0 # Average elo rating of Team B
EAi = 0 # Individual expectation value of player Aifor a given match—indicates the chances of suc-cess for the player and is calculated from RAiand RB
MAi = 0 # Individual number of minutes played byAiin a given match—indicates how long theplayer was on the pitch
DAi = 0 # Individual goal difference of Aifor a givenmatch—indicates the difference of the goals ofthe teams that were scored while the player wason the pitch
Mmax= 90 # Maximum number of minutes in a match

In [3]:
elo_initial = 1500 # Initial ELO rating for players
kAi_initial = 32   # The value indicating the strength of the ratingchange of player Aiis initially 32, maximum 40 andminimum 24.
qAi_initial = 1    # qAi, the value that indicates the weight of theindividual change compared to the team change isinitially 1,
                   # maximum 1 and minimum 0.5

In [4]:
S = {"win" : 1,
     "draw" : 0.5,
     "loss" : 0}
player_elos = defaultdict(lambda: 1500)


In [5]:
understat_team_elo_df = pd.read_csv("Data/understat_matches_with_clubelo.csv")

In [6]:
understat_team_elo_df.columns

Index(['id', 'fid', 'h', 'a', 'league_id', 'season', 'h_goals', 'a_goals',
       'team_h', 'team_a', 'league', 'h_xg', 'a_xg', 'h_d', 'h_deep', 'a_deep',
       'a_ppda', 'h_ppda', 'Shots on target per total shots_home',
       'Shots on target per total shots_away', 'date_for_clubelo',
       'Elo_team_h', 'date', 'Elo_team_a'],
      dtype='object')

In [ ]:
understat_roster_df = pd.read_csv(localized_url("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Fifa/Data/understat_fifa_merged_cleaned.csv"))

In [8]:
understat_roster_df.drop(columns=["Season"], inplace=True)
understat_roster_df.rename(columns={"season": "Season"}, inplace=True)

In [9]:
understat_roster_df[understat_roster_df["id"]==83]

,Understat_Roster_ID,goals,own_goals,shots,xG,time,Understat_Player_ID_x,Understat_Team_ID,Understat_Position,Understat_Player_Name,...,FK Acc.,Penalties,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Player URL_y,Player ID
56,57164,0,0,0,0.000000,90,600,72,GK,tim howard,...,0.0,0.0,0.0,78.0,80.0,78.0,75.0,82.0,NaN,NaN
57,57165,0,0,0,0.000000,90,585,72,DR,seamus coleman,...,35.0,60.0,64.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/180216/s%C3%A...,180216.0
58,57167,0,0,0,0.000000,90,586,72,DC,john stones,...,29.0,37.0,27.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/203574/john-s...,203574.0
59,57166,0,0,0,0.000000,90,587,72,DC,phil jagielka,...,31.0,68.0,44.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/53914/phil-ja...,53914.0
60,57168,0,0,0,0.000000,65,909,72,DL,brendan galloway,...,34.0,44.0,32.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/209852/brenda...,209852.0
61,57170,0,0,1,0.097273,90,590,72,DMC,gareth barry,...,78.0,72.0,65.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/6826/gareth-b...,6826.0
62,57169,0,0,0,0.000000,90,589,72,DMC,james mccarthy,...,64.0,60.0,61.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/188253/james-...,188253.0
63,57171,0,0,1,0.010156,79,595,72,AMR,kevin mirallas,...,74.0,75.0,76.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/172175/kevin-...,172175.0
64,57172,1,0,4,0.092212,90,592,72,AMC,ross barkley,...,56.0,65.0,61.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/199189/ross-b...,199189.0
65,57173,0,0,0,0.000000,90,596,72,AML,tom cleverley,...,62.0,72.0,68.0,0.0,0.0,0.0,0.0,0.0,https://www.fifaindex.com/player/189725/tom-cl...,189725.0


In [10]:
understat_roster_df.drop_duplicates(inplace=True)

In [11]:
understat_roster_df[understat_roster_df["id"]==479][["h_a","Understat_Player_Name", "Understat_Team_ID"]]

,h_a,Understat_Player_Name,Understat_Team_ID
11089,h,vito mannone,77
11090,h,donald love,77
11091,h,papy djilobodji,77
11092,h,john o&#039;shea,77
11093,h,patrick van aanholt,77
11094,h,paddy mcnair,77
11095,h,jack rodwell,77
11096,h,adnan januzaj,77
11097,h,adnan januzaj,77
11098,h,lynden gooch,77


In [12]:
understat_team_elo_df = pd.merge(understat_team_elo_df, understat_roster_df[["id", "Season"]], on="id", how="left")

In [13]:
understat_team_elo_df.drop_duplicates(inplace=True)

In [14]:
understat_roster_df.columns

Index(['Understat_Roster_ID', 'goals', 'own_goals', 'shots', 'xG', 'time',
       'Understat_Player_ID_x', 'Understat_Team_ID', 'Understat_Position',
       'Understat_Player_Name', 'h_a', 'yellow_card', 'red_card', 'roster_in',
       'roster_out', 'key_passes', 'assists', 'xA', 'xGChain', 'xGBuildup',
       'positionOrder', 'Understat_Match_Link', 'Best_Match_Name',
       'Match_Score', 'Understat_Match_ID', 'id', 'Season',
       'Understat_Player_ID_y', 'Fifa_Name', 'Player URL_x', 'Player Name',
       'Team Name', 'Version', 'Ball Control', 'Dribbling', 'Slide Tackle',
       'Stand Tackle', 'Aggression', 'Reactions', 'Att. Position',
       'Interceptions', 'Vision', 'Crossing', 'Short Pass', 'Long Pass',
       'Acceleration', 'Stamina', 'Strength', 'Balance', 'Sprint Speed',
       'Agility', 'Jumping', 'Heading', 'Shot Power', 'Finishing',
       'Long Shots', 'Curve', 'FK Acc.', 'Penalties', 'Volleys',
       'GK Positioning', 'GK Diving', 'GK Handling', 'GK Kicking',
    

In [15]:
understat_roster_df.sort_values(by=['id'],inplace=True)

In [16]:
understat_roster_df.drop(columns=['Understat_Player_ID_y'], inplace=True)
understat_roster_df.rename(columns={'Understat_Player_ID_x': 'Understat_Player_ID'}, inplace=True)

In [17]:
# understat_team_elo_df has h_goals and a_goals columns for the home and away teams respectively 
# which represents the totals goals scored for each team in the match ( home and away)
# understat_team_elo_df also has h: home id and a: away id columns which represent the team ids which we can use to match the understat_roster_df
# Let's create a total_goals column for understat_roster_df for match which will be projected on to each player for convenience
# understat_roster_df has a column 'id' which is the match id, we can use this to join with understat_team_elo_df
understat_roster_df = understat_roster_df.merge(
    understat_team_elo_df[['id', 'h', 'a', 'h_goals', 'a_goals']],
    on='id',
    how='left'
).drop_duplicates()
    

In [18]:
# Now we can create a total_goals column for understat_roster_df
# total_goals = h_goals if the player is from the home team, else a_goals
# understat_roster_df['h_a'] shows whether the player is from the home team or away team
understat_roster_df['total_goals'] = np.where(
    understat_roster_df['h_a'] == 'h',
    understat_roster_df['h_goals'],
    understat_roster_df['a_goals']

)

In [19]:
elo_roster_df= understat_roster_df[["id", "Understat_Team_ID", "Understat_Player_ID", "Understat_Player_Name", "Understat_Position", "Season","h", "a", "h_a", "h_goals", "a_goals", "time"]].copy()

In [20]:
elo_roster_df.id.unique()

array([   81,    82,    83, ..., 22303, 22304, 25861])

In [21]:
season_list = understat_roster_df['Season'].unique().tolist()
elo_roster_2015 = elo_roster_df[elo_roster_df['Season'] == 2015].copy()
elo_roster_2016 = elo_roster_df[elo_roster_df['Season'] == 2016].copy()
elo_roster_2017 = elo_roster_df[elo_roster_df['Season'] == 2017].copy()
elo_roster_2018 = elo_roster_df[elo_roster_df['Season'] == 2018].copy()
elo_roster_2019 = elo_roster_df[elo_roster_df['Season'] == 2019].copy()
elo_roster_2020 = elo_roster_df[elo_roster_df['Season'] == 2020].copy()
elo_roster_2021 = elo_roster_df[elo_roster_df['Season'] == 2021].copy()
elo_roster_2022 = elo_roster_df[elo_roster_df['Season'] == 2022].copy()
elo_roster_2023 = elo_roster_df[elo_roster_df['Season'] == 2023].copy()

In [22]:
elo_roster_df[elo_roster_df["id"]==83]

,id,Understat_Team_ID,Understat_Player_ID,Understat_Player_Name,Understat_Position,Season,h,a,h_a,h_goals,a_goals,time
56,83,90,564,heurelho gomes,GK,2015,72,90,a,2,2,90
57,83,90,571,ben watson,Sub,2015,72,90,a,2,2,9
58,83,90,573,odion ighalo,Sub,2015,72,90,a,2,2,14
59,83,90,574,troy deeney,FW,2015,72,90,a,2,2,90
60,83,90,578,juan carlos paredes,Sub,2015,72,90,a,2,2,29
61,83,90,1016,miguel layun,ML,2015,72,90,a,2,2,61
62,83,90,570,valon behrami,MC,2015,72,90,a,2,2,81
63,83,90,572,etienne capoue,MC,2015,72,90,a,2,2,90
64,83,90,577,ikechi anya,MR,2015,72,90,a,2,2,90
65,83,90,568,jose holebas,DL,2015,72,90,a,2,2,90


In [23]:
elo_roster_2015.id.unique()

array([ 81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,
        94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106,
       107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119,
       120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
       133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145,
       146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158,
       159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171,
       172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184,
       185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197,
       198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210,
       211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223,
       224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236,
       237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249,
       250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 26

In [24]:
elo_team_2015 = understat_team_elo_df[understat_team_elo_df['Season'] == 2015].copy()
elo_team_2016 = understat_team_elo_df[understat_team_elo_df['Season'] == 2016].copy()
elo_team_2017 = understat_team_elo_df[understat_team_elo_df['Season'] == 2017].copy()
elo_team_2018 = understat_team_elo_df[understat_team_elo_df['Season'] == 2018].copy()
elo_team_2019 = understat_team_elo_df[understat_team_elo_df['Season'] == 2019].copy()
elo_team_2020 = understat_team_elo_df[understat_team_elo_df['Season'] == 2020].copy()
elo_team_2021 = understat_team_elo_df[understat_team_elo_df['Season'] == 2021].copy()
elo_team_2022 = understat_team_elo_df[understat_team_elo_df['Season'] == 2022].copy()
elo_team_2023 = understat_team_elo_df[understat_team_elo_df['Season'] == 2023].copy()

In [25]:
elo_roster_2015['Understat_Player_ID']

0        560
1        644
2        645
3        649
4        647
        ... 
10587    629
10588    553
10589    556
10590    552
10591    558
Name: Understat_Player_ID, Length: 10592, dtype: int64

In [26]:
team_ID_list = elo_roster_2015['Understat_Team_ID'].unique().tolist()
player_ID_list = elo_roster_2015['Understat_Player_ID'].unique().tolist()
match_id_list = elo_roster_2015['id'].unique().tolist()

In [27]:
team_ID_list

[89,
 82,
 71,
 73,
 90,
 72,
 77,
 75,
 78,
 79,
 84,
 80,
 74,
 86,
 81,
 83,
 87,
 85,
 88,
 76]

max_rai = elo_roster_2023["RAi"].max()
elo_roster_2023[elo_roster_2023["RAi"] == max_rai]

max_RA= elo_roster_2016['New_RA'].max()
elo_roster_2016[elo_roster_2016["New_RA"] == max_RA]

In [28]:
def team_instance_assignment(seasonal_df):
    seasonal_df = seasonal_df.copy()
    team_ID_list = seasonal_df['Understat_Team_ID'].unique().tolist()
    seasonal_df['team_match_instance'] = None

    for team_ID in team_ID_list:
        # Get unique matches for that team
        team_matches = (
            seasonal_df[seasonal_df['Understat_Team_ID'] == team_ID]
            [['id', 'Understat_Team_ID']]
            .drop_duplicates(subset=['id', 'Understat_Team_ID'])
            .sort_values('id')  # Optional: sort by match ID or date if available
        )

        # Enumerate matches
        for i, (match_idx, match_row) in enumerate(team_matches.iterrows(), 1):
            match_id = match_row['id']
            # Assign match instance to all rows in that match/team combo
            mask = (seasonal_df['id'] == match_id) & (seasonal_df['Understat_Team_ID'] == team_ID)
            seasonal_df.loc[mask, 'team_match_instance'] = i
    return seasonal_df


In [29]:
def player_instance_assignment(seasonal_df):
    seasonal_df = seasonal_df.copy()
    player_ID_list = seasonal_df['Understat_Player_ID'].unique().tolist()
    seasonal_df["player_match_instance"] = None
    for player_ID in player_ID_list:
        # Get unique matches for that player
        player_matches = (
            seasonal_df[seasonal_df['Understat_Player_ID'] == player_ID]
            [['id', 'Understat_Player_ID']]
            .drop_duplicates(subset=['id', 'Understat_Player_ID'])
            .sort_values('id')  # Optional: sort by match ID or date if available
        )

        # Enumerate matches
        for i, (match_idx, match_row) in enumerate(player_matches.iterrows(), 1):
            match_id = match_row['id']
            # Assign match instance to all rows in that match/player combo
            mask = (seasonal_df['id'] == match_id) & (seasonal_df['Understat_Player_ID'] == player_ID)
            seasonal_df.loc[mask, 'player_match_instance'] = i
    return seasonal_df

In [30]:
def kai_calculation(seasonal_df):
    seasonal_df = seasonal_df.copy()
    player_ID_list = seasonal_df['Understat_Player_ID'].unique().tolist()
    team_ID_list = seasonal_df['Understat_Team_ID'].unique().tolist()
    seasonal_df["kAi"] = np.nan
    kAi_initial=40
    # kAi = kAi + 0.5 if Ai was not on the pitch at the time of the match
    # kAi = kAi - 0.25 if Ai was on the pitch at the time of the match
    for player_ID in player_ID_list:
        player_df = seasonal_df[seasonal_df['Understat_Player_ID'] == player_ID]
        player_df = player_df.sort_values('player_match_instance')
        prev_kAi = kAi_initial
        prev_team_instance = None

        for idx, row in player_df.iterrows():
            current_team_instance = row['team_match_instance']

            if row['player_match_instance'] == 1:
                kAi = kAi_initial
            else:
                if prev_team_instance is not None:
                    instances_since_last = current_team_instance - prev_team_instance
                    if instances_since_last > 1:
                        kAi = prev_kAi + 0.5 * instances_since_last
                    else:
                        kAi = prev_kAi - 0.25
                else:
                    kAi = prev_kAi  # fallback if somehow missing

            # Clamp to [24, 40]
            kAi = max(24, min(40, kAi))

            # Assign
            seasonal_df.at[idx, 'kAi'] = kAi

            # Update trackers
            prev_kAi = kAi
            prev_team_instance = current_team_instance
    return seasonal_df

        

In [31]:
def qai_calculation(seasonal_df):
    seasonal_df = seasonal_df.copy()
    player_ID_list = seasonal_df['Understat_Player_ID'].unique().tolist()
    team_ID_list = seasonal_df['Understat_Team_ID'].unique().tolist()
    seasonal_df["qAi"] = np.nan
    # qAi = qAi + 0.025 if Ai was not on the pitch at the time of the match
    # qAi = qAi - 0.025 if Ai was on the pitch at the time of the match
    for player_ID in player_ID_list:
        player_df = seasonal_df[seasonal_df['Understat_Player_ID'] == player_ID]
        player_df = player_df.sort_values('player_match_instance')
        prev_qAi = qAi_initial
        prev_team_instance = None

        for idx, row in player_df.iterrows():
            current_team_instance = row['team_match_instance']

            if row['player_match_instance'] == 1:
                qAi = qAi_initial
            else:
                if prev_team_instance is not None:
                    instances_since_last = current_team_instance - prev_team_instance
                    if instances_since_last > 1:
                        qAi = prev_qAi + 0.025 * instances_since_last
                    else:
                        qAi = prev_qAi - 0.025
                else:
                    qAi = prev_qAi  # fallback if somehow missing

            # Clamp to [0.5, 1]
            qAi = max(0.5, min(1, qAi))

            # Assign
            seasonal_df.at[idx, 'qAi'] = qAi

            # Update trackers
            prev_qAi = qAi
            prev_team_instance = current_team_instance
    return seasonal_df

In [32]:
def elo_calculation(seasonal_df):

    # --- Constants ---
    elo_initial = 1500

    w = 1
    Mmax = 90
    home_advantage_elo = 100  # Home advantage in Elo calculation

    seasonal_df = seasonal_df.copy()  # Avoid modifying in place
    seasonal_df = team_instance_assignment(seasonal_df)
    seasonal_df = player_instance_assignment(seasonal_df)
    seasonal_df = kai_calculation(seasonal_df)
    seasonal_df = qai_calculation(seasonal_df)

    # --- Initialize Elo Columns ---
    seasonal_df["RA"] = np.nan
    seasonal_df["RB"] = np.nan
    seasonal_df["RAi"] = np.nan
    seasonal_df["New_RA"] = np.nan
    seasonal_df["New_RAi"] = np.nan
    seasonal_df["EAi"] = np.nan
    seasonal_df["CAi"] = np.nan
    seasonal_df["CA"] = np.nan

    seasonal_df = seasonal_df.sort_values("id").reset_index(drop=True)

    latest_team_elo = {}
    latest_player_elo = {}

    for match_id in seasonal_df["id"].unique():
        match_df = seasonal_df[seasonal_df["id"] == match_id]

        for team_id in match_df["Understat_Team_ID"].unique():
            team_rows = match_df[match_df["Understat_Team_ID"] == team_id]
            opponent_id = team_rows["a"].iloc[0] if team_rows["h_a"].iloc[0] == "h" else team_rows["h"].iloc[0]

            RA = latest_team_elo.get(team_id, elo_initial)
            RB = latest_team_elo.get(opponent_id, elo_initial)

            DAi = team_rows["h_goals"].iloc[0] - team_rows["a_goals"].iloc[0] if team_rows["h_a"].iloc[0] == "h" else team_rows["a_goals"].iloc[0] - team_rows["h_goals"].iloc[0]
            SAi = 1 if DAi > 0 else 0 if DAi < 0 else 0.5

            for idx, row in team_rows.iterrows():
                player_id = row["Understat_Player_ID"]
                RAi = latest_player_elo.get(player_id, elo_initial)
                MAi = row["time"]
                kAi = row["kAi"]
                qAi = row["qAi"]

                EAi = 1 / (1 + 10 ** ((RB - RAi) / 400))
                CAi = w * (SAi - EAi) * (abs(DAi) ** (1/3)) if DAi != 0 else w * (SAi - EAi) * (MAi / Mmax)

                new_RAi = RAi + kAi * CAi * (1 - EAi) * (MAi / Mmax) * qAi
                new_RAi = max(300, min(3000, new_RAi))  # Optional clamping

                seasonal_df.at[idx, "RA"] = RA
                seasonal_df.at[idx, "RB"] = RB
                seasonal_df.at[idx, "RAi"] = RAi
                seasonal_df.at[idx, "EAi"] = EAi
                seasonal_df.at[idx, "CAi"] = CAi
                seasonal_df.at[idx, "New_RAi"] = new_RAi

                latest_player_elo[player_id] = new_RAi

            team_times = seasonal_df.loc[team_rows.index, "time"]
            player_elos = seasonal_df.loc[team_rows.index, "New_RAi"]
            new_RA = (player_elos * team_times).sum() / team_times.sum()
            CA = seasonal_df.loc[team_rows.index, "CAi"].mean()

            seasonal_df.loc[team_rows.index, "New_RA"] = new_RA
            seasonal_df.loc[team_rows.index, "CA"] = CA

            latest_team_elo[team_id] = new_RA

    return seasonal_df


In [33]:
elo_roster_2015 = elo_calculation(elo_roster_2015); elo_roster_2015["Season"]=2015
elo_roster_2016 = elo_calculation(elo_roster_2016); elo_roster_2016["Season"]=2016
elo_roster_2017 = elo_calculation(elo_roster_2017); elo_roster_2017["Season"]=2017
elo_roster_2018 = elo_calculation(elo_roster_2018); elo_roster_2018["Season"]=2018
elo_roster_2019 = elo_calculation(elo_roster_2019); elo_roster_2019["Season"]=2019
elo_roster_2020 = elo_calculation(elo_roster_2020); elo_roster_2020["Season"]=2020
elo_roster_2021 = elo_calculation(elo_roster_2021); elo_roster_2021["Season"]=2021
elo_roster_2022 = elo_calculation(elo_roster_2022); elo_roster_2022["Season"]=2022
elo_roster_2023 = elo_calculation(elo_roster_2023); elo_roster_2023["Season"]=2023

In [34]:
ra_max= elo_roster_2015["New_RA"].max()
elo_roster_2015[elo_roster_2015["New_RA"] == ra_max][["Understat_Player_Name", "Understat_Team_ID", "New_RA"]]

,Understat_Player_Name,Understat_Team_ID,New_RA
9295,hugo lloris,82,1579.177949
9300,dele alli,82,1579.177949
9301,kyle walker,82,1579.177949
9302,eric dier,82,1579.177949
9303,toby alderweireld,82,1579.177949
9304,danny rose,82,1579.177949
9305,erik lamela,82,1579.177949
9306,jan vertonghen,82,1579.177949
9307,mousa dembele,82,1579.177949
9308,harry kane,82,1579.177949


In [35]:
understat_roster_elo_df = pd.concat([
    elo_roster_2015,
    elo_roster_2016,
    elo_roster_2017,
    elo_roster_2018,
    elo_roster_2019,
    elo_roster_2020,
    elo_roster_2021,
    elo_roster_2022,
    elo_roster_2023
], ignore_index=True)

In [36]:
understat_roster_elo_df.groupby("Season")[["New_RAi"]].max()

,New_RAi
Season,
2015,1598.929550
2016,1618.598680
2017,1638.598102
2018,1636.069560
2019,1639.306631
2020,1622.301076
2021,1633.832521
2022,1626.377578
2023,1626.969239


In [37]:
understat_roster_elo_df.groupby("Season")[["New_RAi"]].max().values[0]

array([1598.92954968])

In [38]:
# Show the max Rai and the corresponding player for each season
understat_roster_elo_df.groupby("Season").apply(lambda x: x.loc[x["New_RAi"].idxmax(), ["Understat_Player_Name", "New_RAi"]])

/var/folders/3p/hrcdb8192mbby6r18mgdgb2c0000gn/T/ipykernel_46022/1346645519.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  understat_roster_elo_df.groupby("Season").apply(lambda x: x.loc[x["New_RAi"].idxmax(), ["Understat_Player_Name", "New_RAi"]])


,Understat_Player_Name,New_RAi
Season,,
2015,danny rose,1598.929550
2016,marcos alonso,1618.598680
2017,kyle walker,1638.598102
2018,trent alexander-arnold,1636.069560
2019,mohamed salah,1639.306631
2020,john stones,1622.301076
2021,aymeric laporte,1633.832521
2022,ruben dias,1626.377578
2023,rodri,1626.969239


In [39]:
understat_roster_elo_df.columns

Index(['id', 'Understat_Team_ID', 'Understat_Player_ID',
       'Understat_Player_Name', 'Understat_Position', 'Season', 'h', 'a',
       'h_a', 'h_goals', 'a_goals', 'time', 'team_match_instance',
       'player_match_instance', 'kAi', 'qAi', 'RA', 'RB', 'RAi', 'New_RA',
       'New_RAi', 'EAi', 'CAi', 'CA'],
      dtype='object')

In [40]:
understat_roster_elo_df["Understat_Position"].unique()

array(['GK', 'AMR', 'Sub', 'FW', 'AML', 'AMC', 'DR', 'DMC', 'DL', 'DC',
       'MC', 'MR', 'ML', 'FWL', 'FWR', 'DML', 'DMR'], dtype=object)

In [41]:
# --- Compute Max_RAi per player, position, season, and team ---
mx2 = (
    df.groupby(["Understat_Player_Name", "Understat_Position", "Season", "Understat_Team_ID"])["RAi"]
      .max()
      .reset_index(name="Max_RAi")
)

# --- Counts per player-position overall (not season-specific) ---
cnt2 = (
    df.groupby(["Understat_Player_Name","Understat_Position"])
      .size()
      .reset_index(name="Counts")
)

# --- Merge: each player's per-season max RAi, with overall counts ---
ranked_fine = (
    mx2.merge(cnt2, on=["Understat_Player_Name","Understat_Position"])
       .sort_values(
           ["Understat_Player_Name","Counts","Max_RAi","Understat_Position"],
           ascending=[True, False, False, True]
       )
       .drop_duplicates("Understat_Player_Name")   # one dominant fine position per player
       .rename(columns={"Understat_Position":"Most_Common_Fine"})
)

# --- Map broad position ---
player_max_Rai = ranked_fine.copy()
player_max_Rai["Most_Common_Position"] = player_max_Rai["Most_Common_Fine"].map(position_matching)
player_max_Rai = player_max_Rai[
    ["Understat_Player_Name","Most_Common_Fine","Most_Common_Position",
     "Counts","Max_RAi","Season","Understat_Team_ID"]
].reset_index(drop=True)

# --- Best per broad role (with 2 centre-backs) ---
player_ranked = (
    player_max_Rai.sort_values(
        ["Most_Common_Position","Max_RAi","Counts","Understat_Player_Name"],
        ascending=[True, False, False, True]
    )
    .reset_index(drop=True)
)

player_ranked["_rank"] = player_ranked.groupby("Most_Common_Position").cumcount() + 1

from collections import defaultdict
take_per_pos = defaultdict(lambda: 1, {"Centre-back": 2})

player_ranked["_take"] = player_ranked["Most_Common_Position"].map(lambda p: take_per_pos[p])
best_by_broad = (
    player_ranked[player_ranked["_rank"] <= player_ranked["_take"]]
    .drop(columns=["_rank","_take"])
    .reset_index(drop=True)
)

# best_by_broad now includes season and team_id context


NameError: name 'df' is not defined

In [ ]:
roster_df= pd.read_csv(localized_url("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Understat/Data/understat_roster_1524.csv"))
team_df= pd.read_csv(localized_url("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Understat/Data/understat_match_1524.csv"))

In [ ]:
team_df.columns

In [ ]:
team_indexes_df= team_df[["h", "team_h", "season"]].drop_duplicates()

In [ ]:
team_indexes_df.sort_values(by=["h"])

In [ ]:
best_by_broad = pd.merge(best_by_broad, team_indexes_df, left_on=["Understat_Team_ID", "Season"], right_on=["h", "season"], how="left")

In [ ]:
best_by_broad = best_by_broad.drop_duplicates().drop(columns=["h", "season", "Counts", "Most_Common_Fine", "Understat_Team_ID"]).reset_index(drop=True)

In [ ]:
best_by_broad

In [ ]:
from collections import defaultdict

# Assumes you already have `player_max_Rai` with:
# ["Understat_Player_Name","Most_Common_Fine","Most_Common_Position","Counts","Max_RAi"]

# 1) Sort once for stable, deterministic ranking within each broad position
player_ranked = (
    player_max_Rai.sort_values(
        ["Most_Common_Position", "Max_RAi", "Counts", "Understat_Player_Name"],
        ascending=[True, False, False, True]
    )
    .reset_index(drop=True)
)

# 2) Rank within each broad position
player_ranked["_rank"] = player_ranked.groupby("Most_Common_Position").cumcount() + 1

# 3) Define desired take-per-position (default=1, Centre-back=2)
take_per_pos = defaultdict(lambda: 1, {"Centre-back": 2})

# 4) Filter by per-position target
player_ranked["_take"] = player_ranked["Most_Common_Position"].map(lambda p: take_per_pos[p])
best_by_broad = (
    player_ranked[player_ranked["_rank"] <= player_ranked["_take"]]
    .drop(columns=["_rank", "_take"])
    .reset_index(drop=True)
)

# `best_by_broad` now includes the top 2 Centre-backs and the top 1 for all other positions.


In [ ]:
best_by_broad

In [ ]:
elo_roster_2015.id.unique()

understat_roster_elo_df["New_RA"] = np.where(
    understat_roster_elo_df["h_a"] == "h",
    understat_roster_elo_df["New_RA"] + 100,
    understat_roster_elo_df["New_RA"]
)


In [ ]:
elo_roster_2016[elo_roster_2016["Understat_Player_ID"]== 572][[
    "Understat_Player_Name","player_match_instance", "team_match_instance", "qAi", "kAi", "EAi", "CAi", "RAi", "CA", "RA", "RB", "New_RA", "New_RAi"]].sort_values(by='team_match_instance')

In [ ]:
understat_roster_elo_df[(understat_roster_elo_df["id"]==22274) & (understat_roster_elo_df["h"]==238)][["RA"]]

In [ ]:
nan_analysis(understat_roster_elo_df)

In [ ]:
understat_team_elo_df.id.unique()

In [ ]:
understat_roster_elo_df.id.unique()

In [ ]:
merged = pd.merge(understat_team_elo_df, understat_roster_elo_df[understat_roster_elo_df["h_a"]=="h"][["id","h", "RA"]], on=["id","h"], how='left').drop_duplicates()

In [ ]:
nan_analysis(merged)

In [ ]:
#Rename RA to Seasonal Elo Home
merged = merged.rename(columns={"RA": "Seasonal_Elo_Home"})

In [ ]:
merged

In [ ]:
merged = pd.merge(merged, understat_roster_elo_df[understat_roster_elo_df["h_a"]=="a"][["id","a", "RA"]], on=["id","a"], how='left').drop_duplicates()
merged = merged.rename(columns={"RA": "Seasonal_Elo_Away"})

In [ ]:
merged.drop_duplicates()

In [ ]:
max_clubelo_h= merged["Elo_team_h"].max()
min_clubelo_h= merged["Elo_team_h"].min()
max_clubelo_a= merged["Elo_team_a"].max()
min_clubelo_a= merged["Elo_team_a"].min()


In [ ]:
max_clubelo_h, min_clubelo_h, max_clubelo_a, min_clubelo_a

In [ ]:
clubelo_difference = max_clubelo_a - min_clubelo_a
clubelo_difference

In [ ]:
max_seasonal_elo_h = merged["Seasonal_Elo_Home"].max()
min_seasonal_elo_h = merged["Seasonal_Elo_Home"].min()
max_seasonal_elo_a = merged["Seasonal_Elo_Away"].max()
min_seasonal_elo_a = merged["Seasonal_Elo_Away"].min()
max_seasonal_elo_h, min_seasonal_elo_h, max_seasonal_elo_a, min_seasonal_elo_a

In [ ]:
seasonal_elo_difference = max_seasonal_elo_h - min_seasonal_elo_h
seasonal_elo_difference

In [ ]:
adjusted_home_advantage = seasonal_elo_difference * 100 / clubelo_difference
adjusted_home_advantage

In [ ]:
merged["Seasonal_Elo_Home"] = merged["Seasonal_Elo_Home"] + adjusted_home_advantage

In [ ]:
merged.to_csv("Data/understat_team_elo_with_seasonal_elo.csv", index=False)

In [ ]:
def nan_analysis(df):
    """
    This function takes a DataFrame and returns a DataFrame with the percentage of NaN values for each column.
    Only columns with NaN values are included in the output DataFrame.
    """
    nan_percentage = df.isna().mean() * 100
    nan_percentage = nan_percentage[nan_percentage > 0].reset_index()
    nan_percentage.columns = ['Column Name', 'Percentage of NaN Values']
    nan_percentage = nan_percentage.sort_values(by='Percentage of NaN Values', ascending=False)
    return nan_percentage

In [ ]:
nan_analysis(merged)